# Raster Analysis for Flood Risk Mapping

Companion notebook for the **EcoGeo Tutor** tutorial: *Python: Raster Analysis for Flood Risk*.

Builds a complete flood detection workflow from Sentinel-1 SAR + a SRTM DEM:
speckle filtering, backscatter thresholding, terrain masking, and export -
with no hydrodynamic model required.

**Data you'll need:**
- Sentinel-1 GRD VV backscatter (in dB) — e.g. from [Copernicus Open Access Hub](https://scihub.copernicus.eu) or Google Earth Engine
- SRTM 30m DEM — e.g. from [USGS EarthExplorer](https://earthexplorer.usgs.gov)

Update the file paths below to match your own downloads.


## Setup

In [ ]:
!pip install numpy rasterio matplotlib scipy -q


## 1. Load data

In [ ]:
import numpy as np
import rasterio
import rasterio.plot
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy.ndimage import uniform_filter

def load_raster(path):
    with rasterio.open(path) as src:
        return src.read(1).astype(float), src.profile, src.transform

sar, sar_profile, sar_transform = load_raster("sentinel1_vv_db.tif")
dem, dem_profile, _             = load_raster("srtm_30m.tif")

print(f"SAR shape: {sar.shape}, range: {sar.min():.1f} - {sar.max():.1f} dB")
print(f"DEM shape: {dem.shape}, range: {dem.min():.0f} - {dem.max():.0f} m")


## 2. Speckle filter (Lee filter)

Speckle is random noise inherent to SAR. Filtering smooths it without losing the flood signal.

In [ ]:
def lee_filter(img, size=7):
    img_mean    = uniform_filter(img, size)
    img_sq_mean = uniform_filter(img**2, size)
    variance    = img_sq_mean - img_mean**2
    weight      = variance / (variance + np.var(img) + 1e-10)
    return img_mean + weight * (img - img_mean)

sar_filtered = lee_filter(sar, size=7)


## 3. Histogram inspection and thresholding

Open water returns very low SAR backscatter. Inspect the histogram before picking a threshold - do not assume a fixed value works for every scene.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(sar_filtered.flatten(), bins=300, color="#374151", alpha=0.7, label="VV backscatter")
ax.axvline(-17, color="#EF4444", lw=2, label="Threshold: -17 dB")
ax.set_xlabel("Backscatter (dB)"); ax.set_ylabel("Pixel count")
ax.set_title("Sentinel-1 VV Backscatter - choose threshold at valley between water and land peaks")
ax.legend(); plt.tight_layout(); plt.show()

THRESHOLD_DB = -17.0
water_sar = sar_filtered < THRESHOLD_DB
print(f"Water pixels detected: {water_sar.sum():,} ({water_sar.sum()/water_sar.size*100:.1f}%)")


## 4. DEM-derived terrain masks

Floodwater follows gravity - it cannot exist on steep slopes or at high elevations. This removes false positives from radar shadow and smooth non-water surfaces.

In [ ]:
pixel_size_m = abs(sar_profile["transform"][0]) * 111000  # approx degrees to metres
dz_dy, dz_dx = np.gradient(dem, pixel_size_m, pixel_size_m)
slope_deg    = np.degrees(np.arctan(np.sqrt(dz_dx**2 + dz_dy**2)))

flat_mask      = slope_deg < 5.0
elev_threshold = np.percentile(dem[dem > 0], 15)
low_mask       = dem < elev_threshold

print(f"Elevation threshold: {elev_threshold:.0f} m")
print(f"Flat pixels (<5deg):   {flat_mask.sum():,}")
print(f"Low-elev pixels:     {low_mask.sum():,}")


## 5. Combine masks, clean, and calculate area

In [ ]:
flood_mask = water_sar & flat_mask & low_mask

# Morphological cleaning removes small isolated false-positive pixels
from scipy.ndimage import binary_opening, binary_closing
flood_clean = binary_closing(binary_opening(flood_mask, iterations=2), iterations=2)

pixel_area_km2 = (30 * 30) / 1e6
flood_area_km2 = flood_clean.sum() * pixel_area_km2
print(f"Estimated flood extent: {flood_area_km2:.1f} km2")


## 6. Visualize the full analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 11))

axes[0,0].imshow(sar_filtered, cmap="gray", vmin=-25, vmax=0)
axes[0,0].set_title("Sentinel-1 VV Backscatter", fontweight="bold"); axes[0,0].axis("off")

im_dem = axes[0,1].imshow(dem, cmap="terrain")
axes[0,1].set_title("SRTM 30m - Elevation (m)", fontweight="bold"); axes[0,1].axis("off")
plt.colorbar(im_dem, ax=axes[0,1], fraction=0.046, pad=0.04)

im_slope = axes[1,0].imshow(slope_deg, cmap="hot_r", vmax=30)
axes[1,0].set_title("Slope (deg) - masked >5deg", fontweight="bold"); axes[1,0].axis("off")
plt.colorbar(im_slope, ax=axes[1,0], fraction=0.046, pad=0.04)

flood_cmap = mcolors.ListedColormap(["#F0F9FF", "#1D4ED8"])
axes[1,1].imshow(flood_clean.astype(int), cmap=flood_cmap, vmin=0, vmax=1)
axes[1,1].set_title(f"Flood Extent - {flood_area_km2:.1f} km2", fontweight="bold")
axes[1,1].axis("off")

plt.suptitle("SAR + DEM Flood Mapping - Sentinel-1 + SRTM", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("flood_analysis_panel.png", dpi=150, bbox_inches="tight")
plt.show()


## 7. Export

In [ ]:
out_profile = sar_profile.copy()
out_profile.update(dtype=rasterio.uint8, count=1, nodata=255)

with rasterio.open("flood_extent.tif", "w", **out_profile) as dst:
    dst.write(flood_clean.astype(np.uint8), 1)

print("Saved: flood_extent.tif")


## Limitations

- **No hydrodynamic flow** — this detects where water is, not how it got there or where it will go. For prediction, integrate HEC-RAS or LISFLOOD-FP.
- **Threshold sensitivity** — the optimal dB threshold varies by scene, sensor mode, and surface conditions.
- **Flooded vegetation** — dense canopy double-bounce can raise backscatter above the threshold, causing flooded forests to be missed.
- **No time evolution** — a single post-event image captures one moment; multi-temporal analysis reveals onset, peak, and recession.

---
*Companion notebook for the EcoGeo Tutor tutorial on rcafe.vercel.app*
